# Feature Engineering

En este notebook trata de mejorar las features de dataset, vamos a trabajar con algunas features para agregar mayor valor al dataset.

## Librerías

In [1]:
import os
import json
import pandas as pd
import numpy as np


In [2]:
config_temporadas = {
    2024: {
        'semana_santa': [('2024-03-24', '2024-03-31')],
        'invierno': [('2024-07-01', '2024-07-31')],
        'especiales': [('2024-11-18', '2024-11-18')] # Feriados puente
    },
    2025: {
        'semana_santa': [('2025-03-01', '2025-03-04')],
        'invierno': [('2025-07-14', '2025-07-27')],
        'verano': [('2025-12-21', '2025-12-31')]
    }
}
mapa_pico = {
    'verano': 1,
    'invierno': 1,
    'semana_santa': 1,
    'especiales': 1,
    'normal': 0
}

In [7]:
feriados = []


def cargar_datos(file_path, **kwargs):
    """
    Carga un archivo CSV o Parquet y devuelve un DataFrame de pandas.

    :param file_path: Ruta al archivo.
    :param kwargs: Argumentos adicionales para pd.read_csv / pd.read_parquet.
    :return: DataFrame cargado.
    :raises ValueError: Si la extensión no es .csv ni .parquet.
    :raises RuntimeError: Si el archivo no se puede leer por otro motivo.
    """
    _, extension_archivo = os.path.splitext(file_path)
    extension_archivo = extension_archivo.lower()

    if extension_archivo == ".csv":
        lector = pd.read_csv
    elif extension_archivo == ".parquet":
        lector = pd.read_parquet
    else:
        raise ValueError(f"Formato no soportado: {extension_archivo}. Usá CSV o Parquet.")

    try:
        return lector(file_path, **kwargs)
    except Exception as e:
        raise RuntimeError(f"Error al cargar el archivo '{file_path}': {e}") from e


def asignar_temporada(fecha):
    anio = str(fecha.year)

    # Si el año no está en nuestro JSON, devolvemos normal
    if anio not in config_temporadas:
        return 'normal'

    # Convertimos la fecha actual a timestamp para comparar rangos
    fecha_dt = pd.to_datetime(fecha)

    for etiqueta, rangos in config_temporadas[anio].items():
        for inicio, fin in rangos:
            if pd.to_datetime(inicio) <= fecha_dt <= pd.to_datetime(fin):
                return etiqueta

    return 'normal'

def clasificar_franja_horaria(hora):
    if 6 <= hora < 12:
        return 'mañana'
    elif 12 <= hora < 18:
        return 'tarde'
    elif 18 <= hora < 23:
        return 'noche'
    else:
        return 'madrugada'


In [4]:
df = cargar_datos("vuelos_historicos_consolidado.parquet")

In [5]:
df.dtypes

,0
Vuelo,object
Ruta,object
hora_programada,object
Hora Real,object
demora_cruda,object
fecha,datetime64[ns]
mes,int64
empresa,object
minutos_netos_demora,float64
estado,object


## Features Temporales

In [9]:
df['dia_semana']    = df['fecha_hora_programada'].dt.dayofweek      # 0=Lunes, 6=Domingo
df['dia_mes']       = df['fecha_hora_programada'].dt.day
df['semana_anio']   = df['fecha_hora_programada'].dt.isocalendar().week
df['es_fin_de_semana'] = df['dia_semana'].isin([4, 5, 6]).astype(int)
df['hora']          = df['fecha_hora_programada'].dt.hour
df
# Features derivadas con significado de negocio
# Viernes también se incluye porque los vuelos del viernes tarde se comportan como fin de semana

df['franja_horaria'] = df['hora_del_dia'].apply(clasificar_franja_horaria)

# Vuelo en hora pico (aeropuertos más congestionados entre 7-9 y 17-20)
df['es_hora_pico'] = df['hora_del_dia'].apply(lambda h: 1 if (7 <= h <= 9 or 17 <= h <= 20) else 0)

# Encoding cíclico de mes y hora: un entero lineal hace que diciembre (12) y
# enero (1) parezcan "lejos", cuando en realidad son consecutivos. El seno/coseno
# preserva esa continuidad para modelos que lo puedan aprovechar.
df['mes_sin']  = np.sin(2 * np.pi * df['mes'] / 12)
df['mes_cos']  = np.cos(2 * np.pi * df['mes'] / 12)
df['hora_sin'] = np.sin(2 * np.pi * df['hora_del_dia'] / 24)
df['hora_cos'] = np.cos(2 * np.pi * df['hora_del_dia'] / 24)

In [10]:
df[['origen', 'destino']] = df['Ruta'].str.split('→', expand=True)
df['origen'] = df['origen'].str.strip()
df['destino'] = df['destino'].str.strip()


## Temporada Alta

Feriados

In [12]:
from datetime import date
import holidays

ar_holidays = holidays.Argentina()

def es_feriado(fecha):
    return True if fecha in ar_holidays else False


anios_presentes = df['fecha'].dt.year.unique().tolist()
# Sumamos el año siguiente al último presente: si no, las fechas de fin de diciembre
# quedan sin 'próximo feriado' (el 1 de enero del año que viene) y dan NaN.
anios_a_incluir = anios_presentes + [max(anios_presentes) + 1]
feriados_ar = holidays.Argentina(years=anios_a_incluir)

df['es_feriado'] = df['fecha'].dt.date.isin(feriados_ar).astype(int)




In [13]:
fechas_feriados = pd.DataFrame({
    'fecha_feriado': sorted(pd.to_datetime(list(feriados_ar.keys())))
})
# Alineamos la resolución de ambas columnas datetime; si no, merge_asof falla
# con un MergeError por dtypes incompatibles (ej. datetime64[us] vs datetime64[s]).
fechas_feriados['fecha_feriado'] = fechas_feriados['fecha_feriado'].astype('datetime64[ns]')

df_ordenado = df.sort_values('fecha').reset_index(drop=False)
df_ordenado['fecha'] = df_ordenado['fecha'].astype('datetime64[ns]')

feriado_anterior = pd.merge_asof(
    df_ordenado[['fecha']], fechas_feriados,
    left_on='fecha', right_on='fecha_feriado', direction='backward'
)
feriado_siguiente = pd.merge_asof(
    df_ordenado[['fecha']], fechas_feriados,
    left_on='fecha', right_on='fecha_feriado', direction='forward'
)

df_ordenado['dias_desde_feriado'] = (df_ordenado['fecha'] - feriado_anterior['fecha_feriado']).dt.days
df_ordenado['dias_hasta_feriado'] = (feriado_siguiente['fecha_feriado'] - df_ordenado['fecha']).dt.days

df = df_ordenado.set_index('index').sort_index()
df.index.name = None


In [15]:
# Aplicación
df["temporada"] = df["fecha"].apply(asignar_temporada)
df['es_pico_demanda'] = df['temporada'].map(mapa_pico)


In [16]:
df[df["temporada"] == "invierno"].tail()


,Vuelo,Ruta,hora_programada,Hora Real,demora_cruda,fecha,mes,empresa,minutos_netos_demora,estado,...,mes_cos,hora_sin,hora_cos,origen,destino,es_feriado,dias_desde_feriado,dias_hasta_feriado,temporada,es_pico_demanda


In [17]:
fechas_paro = ["2025-04-10", "2025-08-22", "2025-12-17"]      # Paros generales / de controladores
fechas_clima_adverso = ["2025-07-08"]                          # Niebla en aeropuertos de Buenos Aires

df["es_paro"] = df["fecha"].isin(pd.to_datetime(fechas_paro)).astype(int)
df["es_evento_climatico"] = df["fecha"].isin(pd.to_datetime(fechas_clima_adverso)).astype(int)


In [ ]:
# completar